In [ ]:
import os

os.environ["HF_HOME"] = "/path/to/hf"
os.environ["HF_HUB_CACHE"] = "/path/to/hf"
os.environ["SENTENCE_TRANSFORMERS_HOME"] = "/path/to/hf"

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from glob import glob
from sentence_transformers import CrossEncoder

models = glob("../helmBenchmark/*.pkl")

In [ ]:
NLI = CrossEncoder('cross-encoder/nli-deberta-v3-large', device='cuda')

In [ ]:
def compute_scores(df): #computes scores for each ref-pred pair (and pred-ref pair)
    temp=[]
    for idx, row in df.iterrows():
        f1 = row['stats_f1_score']
        bleu = row['stats_bleu_4']
    
        if not pd.isna(f1) or not pd.isna(bleu):
            pred = row['predicted_text']
            refs = row['references']
            
            scores=[]
            for ref in refs:
                ref = ref['output']['text']
                score = NLI.predict([(pred, ref), (ref, pred)])
                scores.append(score)
            temp.append(scores)
        else:
            temp.append([])
    return temp

def get_labels(score_list):
    l=[]
    for scores in score_list: #scores represents the list of scores in a row
        if len(scores) > 0: #not empty (f1 / bleu score exist)
            temp=[]
            for score in scores:
                label_mapping = ['contradiction', 'entailment', 'neutral']
                labels = [label_mapping[score_max] for score_max in score.argmax(axis=1)] 
                temp.append(labels)
            l.append(temp)
        else:
            l.append([])
    return l

def get_entailment_model_decision(labels):
    Vee, Eee, Ven, Een = [], [], [], []
    
    for line in labels:
        if len(line) > 0:
    
            vee_flag = True
            eee_flag = False
            ven_flag = True
            een_flag = False
    
            for element in line:
                if element != ['entailment', 'entailment']:
                    vee_flag = False
                if element == ['entailment', 'entailment']:
                    eee_flag = True
                if element not in [['entailment', 'entailment'], ['entailment', 'neutral'], ['neutral', 'entailment']]:
                    ven_flag = False
                if element in [['entailment', 'entailment'], ['entailment', 'neutral'], ['neutral', 'entailment']]:
                    een_flag = True
    
            Vee.append(1 if vee_flag else 0)
            Eee.append(1 if eee_flag else 0)
            Ven.append(1 if ven_flag else 0)
            Een.append(1 if een_flag else 0)
        else:
            Vee.append(np.nan)
            Eee.append(np.nan)
            Ven.append(np.nan)
            Een.append(np.nan)

    return Vee, Eee, Ven, Een

In [ ]:
for model in models:
    df = pd.read_pickle(model)
    scores = compute_scores(df)
    labels = get_labels(scores)
    Vee, Eee, Ven, Een = get_entailment_model_decision(labels)

    df['Vee'] = Vee
    df['Eee'] = Eee
    df['Ven'] = Ven
    df['Een'] = Een
    
    df.to_pickle(model)